# 04 — Hvordan LLMs faktisk fungerer

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 01–03

**Hva du bygger:** Praktiske eksperimenter som viser tokenisering, kontekstvinduer og temperatur — slik at du slutter å behandle LLMs som magiske bokser.

---

## Én setning om hva en LLM gjør

> En LLM tar en sekvens av tokens og beregner sannsynligheten for hvert mulige neste token — deretter velger den ett, og gjentar.

Det er alt. Ingen "forståelse", ingen "tenkning" — bare statistikk over enorme mengder tekst. Resultatet er likevel imponerende nyttig.

In [ ]:
%pip install -q tiktoken

---

## Del 1: Tokenisering i praksis

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

def vis_tokens(tekst: str):
    tokens = enc.encode(tekst)
    biter  = [enc.decode([t]).replace(' ', '·') for t in tokens]
    print(f"Tekst:    {tekst}")
    print(f"Tokens:   {biter}")
    print(f"Antall:   {len(tokens)}")
    print()

vis_tokens("Hei")
vis_tokens("pensjonsordning")
vis_tokens("Statens pensjonskasse")
vis_tokens("AI Engineer")

In [ ]:
# Tokenisering er språkavhengig — norsk er dårligere dekket enn engelsk
# Dette påvirker kostnad og kontekstvindu-utnyttelse

setning_no = "AFP gir deg rett til å gå av med pensjon fra 62 år."
setning_en = "AFP gives you the right to retire with a pension from age 62."

tokens_no = enc.encode(setning_no)
tokens_en = enc.encode(setning_en)

print(f"Norsk ({len(tokens_no)} tokens): {setning_no}")
print(f"Engelsk ({len(tokens_en)} tokens): {setning_en}")
print(f"\nNorsk bruker {len(tokens_no)/len(tokens_en):.1f}x så mange tokens for samme innhold.")
print("→ Norske prompts koster mer og passer dårligere i kontekstvinduer.")

---

## Del 2: Kontekstvinduer

LLMs har et **kontekstvindu** — en grense for hvor mye tekst de kan behandle på én gang.

| Modell | Kontekstvindu | Ca. sider tekst |
|--------|--------------|----------------|
| GPT-3.5 | 16 000 tokens | ~12 sider |
| GPT-4o | 128 000 tokens | ~96 sider |
| Claude 3.5 Sonnet | 200 000 tokens | ~150 sider |
| Gemini 1.5 Pro | 1 000 000 tokens | ~750 sider |

**Hva passer i konteksten til GPT-4o (128k)?**

In [ ]:
GPT4O_KONTEKST = 128_000  # tokens

def tokens_per_side(ord_per_side=250, tokens_per_ord=1.5):
    return ord_per_side * tokens_per_ord

ting = {
    "Én e-post (kort)": 100,
    "Én side A4-tekst": 400,
    "Pensjonsloven §1-20": 2_000,
    "Kort fagartikkel": 5_000,
    "Hel pensjonsguide (50 sider)": 20_000,
    "En hel roman": 180_000,
}

print(f"{'Innhold':<35} {'Tokens':>8}  {'Passer i GPT-4o?':>18}")
print("-" * 65)
for navn, tokens in ting.items():
    passer = "✅ Ja" if tokens < GPT4O_KONTEKST else "❌ Nei — må chunkes"
    print(f"{navn:<35} {tokens:>8,}  {passer:>18}")

---

## Del 3: Chunking — dele opp lange dokumenter

Når et dokument er for langt for kontekstvindyet (eller for RAG), deler vi det opp i **chunks**.

In [ ]:
def chunk_tekst(tekst: str, chunk_størrelse: int = 200, overlapp: int = 50) -> list[str]:
    """
    Del tekst i chunks av maks chunk_størrelse tokens.
    Overlapp sikrer at setninger ved grenser ikke mister kontekst.
    """
    tokens = enc.encode(tekst)
    chunks = []
    start = 0
    
    while start < len(tokens):
        slutt = min(start + chunk_størrelse, len(tokens))
        chunk_tokens = tokens[start:slutt]
        chunks.append(enc.decode(chunk_tokens))
        start += chunk_størrelse - overlapp  # Flytt med overlapp
    
    return chunks

# Test på en lengre tekst
lang_tekst = """Statens pensjonskasse (SPK) er en av Norges største pensjonsordninger.
SPK forvalter pensjoner for statsansatte, lærere og ansatte i statlige virksomheter.
Ordningen inkluderer alderspensjon, uførepensjon, AFP og etterlattepensjon.
AFP (avtalefestet pensjon) gir mulighet til tidligpensjonering fra 62 år.
Alderspensjon utbetales normalt fra 67 år, men kan tas ut fra 62 med reduksjon.
Uførepensjon gis ved varig nedsatt arbeidsevne med minst 20 prosent.
Barnepensjon utbetales til barn under 20 år når en forsørger dør."""

chunks = chunk_tekst(lang_tekst, chunk_størrelse=50, overlapp=10)

print(f"Total: {len(enc.encode(lang_tekst))} tokens → {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(enc.encode(chunk))} tokens):")
    print(f"  {chunk[:80].strip()}..." if len(chunk) > 80 else f"  {chunk.strip()}")

---

## Del 4: Temperature og sampling

**Temperature** styrer hvor forutsigbar modellen er:
- `0.0` — alltid det mest sannsynlige svaret (deterministisk)
- `0.7` — god balanse kreativitet/presisjon (standard)
- `2.0` — veldig kreativ, kan bli usammenhengende

**Når bruker du hva?**

| Bruk | Temperature | Begrunnelse |
|------|------------|-------------|
| Fakta-spørsmål (pensjon) | 0.0–0.3 | Vil ha presise, konsistente svar |
| Oppsummering | 0.3–0.7 | Litt variasjon OK |
| Kreativ skriving | 0.8–1.2 | Ønsker variasjon |
| Kode-generering | 0.0–0.2 | Kode skal være korrekt, ikke kreativ |

---

## Miniprosjekt: Token-budsjettkalkulator

Bygg et verktøy som estimerer kostnad og feasibility for en LLM-oppgave.

In [ ]:
from dataclasses import dataclass

@dataclass
class LLMModell:
    navn: str
    kontekst_tokens: int
    pris_input_per_1k: float   # USD
    pris_output_per_1k: float  # USD

MODELLER = [
    LLMModell("gpt-4o",               128_000, 0.0025, 0.010),
    LLMModell("gpt-4o-mini",          128_000, 0.00015, 0.0006),
    LLMModell("claude-sonnet-4-6",    200_000, 0.003,  0.015),
    LLMModell("claude-haiku-4-5",     200_000, 0.0008, 0.004),
]

def analyser_oppgave(system_prompt: str, bruker_melding: str,
                     forventet_svar_tokens: int = 500,
                     antall_kall_per_dag: int = 1000):
    input_tokens = len(enc.encode(system_prompt + bruker_melding))
    
    print(f"{'Modell':<22} {'Passer?':>8} {'Per kall':>10} {'Per dag':>10}")
    print("-" * 55)
    
    for m in MODELLER:
        passer = "✅" if input_tokens + forventet_svar_tokens < m.kontekst_tokens else "❌"
        kostnad_kall = (input_tokens / 1000 * m.pris_input_per_1k +
                        forventet_svar_tokens / 1000 * m.pris_output_per_1k)
        kostnad_dag = kostnad_kall * antall_kall_per_dag
        print(f"{m.navn:<22} {passer:>8} ${kostnad_kall:>8.4f} ${kostnad_dag:>8.2f}")

# Eksempel: SPK pensjonschatbot
system = "Du er en pensjonsrådgiver hos SPK. Svar kort og presist på norsk."
spørsmål = "Hva er forskjellen mellom AFP og alderspensjon?"

print(f"System-prompt: {len(enc.encode(system))} tokens")
print(f"Spørsmål: {len(enc.encode(spørsmål))} tokens\n")
analyser_oppgave(system, spørsmål, forventet_svar_tokens=300, antall_kall_per_dag=5000)

---

## Oppsummering

- LLMs genererer tekst token for token, basert på sannsynligheter
- Norsk tekst bruker ~1.3–1.5x så mange tokens som tilsvarende engelsk
- Kontekstvindyet er hardt — store dokumenter må chunkes
- Temperature = 0 for fakta, høyere for kreativitet
- Velg modell basert på kontekstbehov + kostnad

---

## Hva er neste steg?

**Neste: `05_llm_apis.ipynb`** — Nå ringer vi faktisk opp en LLM. Du lærer chat-API-et, streaming, tool calling og prompt engineering med ekte kode.